# **Objectif**
### L'objectif est de faire une analyse claire des avis clients et de proposer des recommandations business.

# **Compréhension des données**

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("abdallahwagih/amazon-reviews")

print("Path to dataset files:", path)

100%|██████████| 44.3M/44.3M [00:00<00:00, 110MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/abdallahwagih/amazon-reviews/versions/1


In [3]:
path

'/root/.cache/kagglehub/datasets/abdallahwagih/amazon-reviews/versions/1'

In [4]:
df=pd.read_json('/root/.cache/kagglehub/datasets/abdallahwagih/amazon-reviews/versions/1/Cell_Phones_and_Accessories_5.json', lines=True)

In [5]:
df.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime
0,A30TL5EWN6DFXT,120401325X,christina,"[0, 0]",They look good and stick good! I just don't li...,4,Looks Good,1400630400,"05 21, 2014"
1,ASY55RVNIL0UD,120401325X,emily l.,"[0, 0]",These stickers work like the review says they ...,5,Really great product.,1389657600,"01 14, 2014"
2,A2TMXE2AFO7ONB,120401325X,Erica,"[0, 0]",These are awesome and make my phone look so st...,5,LOVE LOVE LOVE,1403740800,"06 26, 2014"
3,AWJ0WZQYMYFQ4,120401325X,JM,"[4, 4]",Item arrived in great time and was in perfect ...,4,Cute!,1382313600,"10 21, 2013"
4,ATX7CZYFXI1KW,120401325X,patrice m rogoza,"[2, 3]","awesome! stays on, and looks great. can be use...",5,leopard home button sticker for iphone 4s,1359849600,"02 3, 2013"


In [6]:
df.shape

(194439, 9)

In [7]:
reviews=df['reviewText']

Le dataset contient les avis clients de 194439 clients d'Amazon. La colonne qui a fait l'objet de notre analyse est la colonne **reviewText**.

# **Nettoyage des données**

In [8]:
import re
def clean(text):
  text=text.lower()
  text=re.sub(r'[^a-zA-Z ]', '', text)
  return text

In [9]:
df['reviews_cl']= df['reviewText'].apply(clean)

# **Analyse des sentiments**

In [10]:
from textblob import TextBlob
def get_sentiment(text):
  score=TextBlob(text).sentiment.polarity
  if score > 0:
    return 'avis positifs'
  elif score < 0:
    return 'avis négatifs'
  else:
    return 'avis neutres'

In [11]:
df['sentiment']=df['reviews_cl'].apply(get_sentiment)

In [12]:
sentiment=df['sentiment'].value_counts()

In [13]:
sentiment = pd.DataFrame(sentiment).reset_index()

In [14]:
import plotly.express as px
fig=px.pie(names='sentiment', values='count', data_frame=sentiment, title='Proportion de chaque sentiment')
fig.show()

L'analyse des avis clients montre que les clients ont en grande majorité des avis positifs (88.4%). Ce qui signifie que les clients sont en générale, satisfaits des services de l'entreprise. Nous mènerons une analyse plus approfondie des mots clés afin de comprendre pourquoi les satisfaits et aussi pourquoi d'autres non.

# **Mots fréquents**

In [17]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

In [18]:
from nltk.corpus import stopwords
from collections import Counter

stop_words = set(stopwords.words('english'))

# Re-create words list, filtering out stopwords
cleaned_words = [word for word in "".join(df['reviews_cl']).split() if word not in stop_words]
freq = Counter(cleaned_words)

In [21]:
freq.most_common(20)

[('phone', 169106),
 ('case', 140405),
 ('one', 83590),
 ('like', 72181),
 ('great', 60815),
 ('use', 60503),
 ('screen', 59712),
 ('battery', 56143),
 ('good', 55721),
 ('would', 54722),
 ('iphone', 48240),
 ('well', 47390),
 ('get', 46724),
 ('charge', 44478),
 ('really', 37931),
 ('charger', 37384),
 ('also', 36131),
 ('dont', 34480),
 ('time', 34357),
 ('product', 33918)]

In [30]:
cleaned_neg_words = [word for word in "".join(négatifs).split() if word not in stop_words]
freq_neg = Counter(cleaned_neg_words)

In [31]:
freq_neg.most_common(10)

[('phone', 11144),
 ('case', 9273),
 ('one', 5562),
 ('like', 4872),
 ('screen', 4241),
 ('get', 3830),
 ('would', 3767),
 ('use', 3638),
 ('battery', 2957),
 ('well', 2780)]

L’analyse des avis négatifs met en évidence une concentration des insatisfactions autour de certains produits, notamment les téléphones et les coques, suggérant que ces catégories sont particulièrement exposées aux problèmes clients. Les termes associés à des composants comme l’écran et la batterie indiquent des préoccupations récurrentes liées à la durabilité et aux performances. Par ailleurs, certains indices linguistiques suggèrent des problèmes de fonctionnement ou de compatibilité, nécessitant une analyse qualitative plus approfondie pour identifier précisément les causes.

## **Thèmes clés**

L’analyse des mots fréquents fait ressortir trois thèmes principaux : **la qualité du produit, l’expérience utilisateur et la perception de la marque (notamment iPhone)**.

In [23]:
négatifs=df['reviews_cl'][df['sentiment']=='avis négatifs']

# **Insights**

L’analyse des avis clients révèle une **satisfaction globale élevée (88.4% d'avis positifs)**, principalement attribuée à la qualité des produits et à l'expérience utilisateur, avec une mention particulière pour les produits Apple.

Cependant, une analyse approfondie des **avis négatifs (17.5%)** a permis d'identifier des points de friction clés:

*   **Catégories de produits:** Les `téléphones` et les `coques` sont des articles fréquemment mentionnés dans les avis négatifs, ce qui indique qu'ils sont des sources courantes de problèmes.
*   **Composants spécifiques:** Des mots comme `écran` et `batterie` ressortent, suggérant des préoccupations liées à la durabilité ou aux performances de ces composants.
*   **Fonctionnalité/Qualité:** Le terme `bien` (souvent utilisé dans des contextes négatifs comme 'pas bien' ou 'ne fonctionne pas bien') indique des soucis de fonctionnalité ou de qualité générale des produits.

### **Recommandations Business :**

1.  **Capitaliser sur les points forts :** Continuer à mettre en avant la qualité et l'expérience utilisateur positive, particulièrement pour les produits phares comme ceux d'Apple.
2.  **Améliorer les points faibles :** Mener une enquête plus approfondie sur les problèmes liés aux `écrans` et aux `batteries`, en particulier pour les `téléphones` et `coques`, pour identifier les causes sous-jacentes et mettre en œuvre des améliorations de conception ou de fabrication.
3.  **Suivi des mentions négatives :** Mettre en place un système de surveillance des avis pour détecter rapidement les tendances négatives autour des mots clés identifiés (`écran`, `batterie`, `téléphone`, `coque`) afin de réagir proactivement.
4.  **Optimisation de la communication :** Clarifier les descriptions de produits pour éviter les déceptions liées à la fonctionnalité ou à la compatibilité (`bien` vs 'pas bien').

Ces actions permettront à l'entreprise de renforcer la satisfaction client et de transformer les points faibles en opportunités d'amélioration.